In [7]:
from tdmpc2lora_tmp.config import Config
from tdmpc2lora_tmp.train_env import make_env
from tdmpc2lora_tmp.model import TDMPC2 # model.pyにリネーム済みと仮定
from tdmpc2lora_tmp.trainer import OnlineTrainer
DO_TEST_0 = False
if DO_TEST_0:
    # 実験設定
    cfg = Config()
    cfg.task = "SimpleReacher"
    cfg.task_id = 0
    cfg.lora_rank = 4 # LoRA有効化

    # 環境・エージェント作成
    env = make_env(cfg)
    agent = TDMPC2(cfg)

    # 学習開始
    trainer = OnlineTrainer(cfg, env, agent)
    trainer.train()

----
----
----

In [ ]:
import torch
from tdmpc2lora_tmp.config import Config
from tdmpc2lora_tmp.train_env import make_env
from tdmpc2lora_tmp.model import TDMPC2
from tdmpc2lora_tmp.trainer import OnlineTrainer

def train_task(task_id, lora_rank, prev_model_path=None):
    print(f"\n=== Training Task {task_id} (LoRA Rank: {lora_rank}) ===")
    
    # 1. 設定
    cfg = Config()
    cfg.task = "SimpleReacher"
    cfg.task_id = task_id
    cfg.num_tasks = 4
    cfg.lora_rank = lora_rank
    cfg.steps = 5000 # 確認ようなので短く
    
    # 実験名に工夫を入れる（Task0 -> Task1 の流れがわかるように）
    base_name = f"Reacher_Task{task_id}_rank{lora_rank}"
    if prev_model_path:
        base_name += "_continued"
    # Configのrun_nameをハックして上書き（簡易的な方法）
    cfg.__class__.run_name = property(lambda self: base_name)

    # 2. 環境・エージェント作成
    env = make_env(cfg)
    agent = TDMPC2(cfg)
    # print("--- Parameter Names Check ---")
    # for name, param in agent.model.named_parameters():
    #     # LoRAパラメータかどうかを判定して表示
    #     is_lora = "lora" in name
    #     print(f"{'[LoRA]' if is_lora else '[Base]'} {name}")

    # 3. 過去のモデルのロード（継続学習の場合）
    if prev_model_path:
        print(f"Loading model from: {prev_model_path}")
        # CPUでロード
        ckpt = torch.load(prev_model_path, map_location="cpu", weights_only=False)
        
        # --- 重要: 重みのロード処理 ---
        # 形状が違う（LoRAが増える等）とエラーになるので、strict=Falseでロードし
        # 共通部分（Core）だけを復元する挙動を確認する
        keys = agent.model.load_state_dict(ckpt["model"], strict=False)
        print(f"Loaded keys: {len(keys.missing_keys)} missing, {len(keys.unexpected_keys)} unexpected")
        
        # 共通部分（LoRA以外）を固定するかどうかのロジックもここでテスト
        for name, param in agent.model.named_parameters():
            if "lora" not in name:
                param.requires_grad = False
            #     print(f"[Base] Param: {name}, requires_grad={param.requires_grad}")
            # else:
            #     print(f"[LoRA] Param: {name}, requires_grad={param.requires_grad}")
    
    # 4. 学習開始
    trainer = OnlineTrainer(cfg, env, agent)
    trainer.train()
    
    # 保存されたベストモデルのパスを返す
    return cfg.get_model_dir() / "best.pth"

# --- 実験フロー ---
if __name__ == "__main__":
    # Phase 1: Task 0 を学習 (LoRAなし または あり)
    model_path_task0 = train_task(task_id=0, lora_rank=4, prev_model_path=None)
    
    # # Phase 2: Task 1 を学習 (Task 0 のモデルを引き継ぐ)
    # # ここでエラーが出なければ、継続学習のパイプラインは完成です！
    train_task(task_id=1, lora_rank=4, prev_model_path=model_path_task0)


=== Training Task 0 (LoRA Rank: 4) ===
[Env] Created SimpleReacherEnv for Task 0 (Target: [1. 1.])
[Logger] Logs will be saved to: result/logs/Reacher_Task0_rank4
[Logger] Models will be saved to: result/models/Reacher_Task0_rank4
Start training on SimpleReacher (Mac Local / CPU)
Check results in: result
Step: 1000, Loss: 102.719
Eval at step 1000: Reward -180.9
 [CheckPoint] New best model saved at step 1000
Step: 2000, Loss: 0.391
Eval at step 2000: Reward -139.2
 [CheckPoint] New best model saved at step 2000
Step: 3000, Loss: 0.358
Eval at step 3000: Reward -123.9
 [CheckPoint] New best model saved at step 3000
Step: 4000, Loss: 0.353
Eval at step 4000: Reward -84.4
 [CheckPoint] New best model saved at step 4000
Step: 5000, Loss: 0.354
Eval at step 5000: Reward -95.7

=== Training Task 1 (LoRA Rank: 4) ===
[Env] Created SimpleReacherEnv for Task 1 (Target: [-1. -1.])
Loading model from: result/models/Reacher_Task0_rank4/best.pth
Loaded keys: 0 missing, 0 unexpected
[LoRA] Param: 

In [5]:
import torch
import numpy as np
from pathlib import Path
from tdmpc2lora_tmp.config import Config
from tdmpc2lora_tmp.train_env import make_env
from tdmpc2lora_tmp.model import TDMPC2
from tdmpc2lora_tmp.trainer import OnlineTrainer

# --- 共通設定 ---
NUM_TASKS = 4  # タスクの総数 (A, B, C, D)
LORA_RANK = 4
DEBUG_STEPS = 5000
# DEBUG_STEPS = 1000 # ローカル確認用なので短く (本番は 20000~ など)

def run_training_step(task_id, prev_model_path=None, run_suffix=""):
    """1つの学習ステップを実行"""
    task_name = f"Task{task_id}"
    print(f"\n{'='*10} Training {task_name} {run_suffix} {'='*10}")
    
    cfg = Config()
    cfg.task = "SimpleReacher"
    cfg.task_id = task_id
    cfg.num_tasks = NUM_TASKS
    cfg.lora_rank = LORA_RANK
    cfg.steps = DEBUG_STEPS
    
    # 保存名: Reacher_Task0_rank4_step1 などのように履歴を残す
    base_name = f"Reacher_{task_name}_rank{LORA_RANK}{run_suffix}"
    cfg.__class__.run_name = property(lambda self: base_name)
    
    # 環境・エージェント
    env = make_env(cfg)
    agent = TDMPC2(cfg)
    
    # モデルロード
    if prev_model_path:
        print(f"Loading weights from: {prev_model_path}")
        ckpt = torch.load(prev_model_path, map_location="cpu", weights_only=False)
        agent.model.load_state_dict(ckpt["model"], strict=False)
        
        # --- Baseパラメータ固定 (LoRAのみ学習) ---
        # Task 0 (初回) 以外はBaseを固定して実験する場合
        # ※ もし「Baseも微調整したい」ならここをコメントアウト
        print("Freezing Base parameters...")
        for name, param in agent.model.named_parameters():
            if "lora" not in name:
                param.requires_grad = False
    
    # 学習
    trainer = OnlineTrainer(cfg, env, agent)
    trainer.train()
    
    return cfg.get_model_dir() / "best.pth"

def evaluate_task(task_id, model_path):
    """特定のタスクIDでモデルを評価"""
    print(f"\n>>> Evaluating Task {task_id} using model: {model_path}")
    
    cfg = Config()
    cfg.task = "SimpleReacher"
    cfg.task_id = task_id # 環境のゴール設定用
    cfg.num_tasks = NUM_TASKS
    cfg.lora_rank = LORA_RANK
    
    env = make_env(cfg)
    agent = TDMPC2(cfg)
    
    # ロード
    ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
    agent.model.load_state_dict(ckpt["model"], strict=False)
    agent.eval()
    
    # 評価ループ
    rewards = []
    for _ in range(5): # 5エピソード平均
        obs, _ = env.reset()
        done = False
        ep_reward = 0
        while not done:
            # task_idx を指定して推論 (ここが重要！)
            # Task 0 の性能を見たいなら task_idx=0 のLoRAを使う
            action = agent.act(obs, eval_mode=True, task_idx=task_id)
            obs, reward, done_tensor, info = env.step(action)
            done = bool(done_tensor.item())
            ep_reward += reward.item()
        rewards.append(ep_reward)
    
    avg_reward = np.mean(rewards)
    print(f"Result Task {task_id}: Average Reward = {avg_reward:.2f}")
    return avg_reward

if __name__ == "__main__":
    # ==========================================
    # 実験フロー: A -> B -> C -> B -> C -> D
    # ==========================================
    
    # 1. Train A (Task 0)
    # 最初は Base も学習される (prev_model_path=None なので固定ロジックが走らない想定、または関数内で分岐が必要)
    # ここでは「初回は全学習、2回目以降はLoRAのみ」とする運用なら関数内の固定ロジックを調整してください。
    # 今回は簡単のため「関数内で prev_model_path があれば固定」としています。
    path_A = run_training_step(0, prev_model_path=None, run_suffix="_init")
    
    # 2. Train B (Task 1) - Aを引き継ぐ
    path_B1 = run_training_step(1, prev_model_path=path_A, run_suffix="_1st")
    
    # 3. Train C (Task 2) - Bを引き継ぐ
    path_C1 = run_training_step(2, prev_model_path=path_B1, run_suffix="_1st")
    
    # 4. Train B (Task 1) - Cを引き継ぐ (忘却の確認や再適応)
    path_B2 = run_training_step(1, prev_model_path=path_C1, run_suffix="_2nd")
    
    # 5. Train C (Task 2) - Bを引き継ぐ
    path_C2 = run_training_step(2, prev_model_path=path_B2, run_suffix="_2nd")
    
    # 6. Train D (Task 3) - Cを引き継ぐ (新規タスク)
    path_D = run_training_step(3, prev_model_path=path_C2, run_suffix="_final")
    
    print("\n" + "="*30)
    print("ALL TRAINING FINISHED. STARTING EVALUATION.")
    print("="*30)
    
    # ==========================================
    # 最終評価: 最後にできたモデルで過去のタスクができるか？
    # ==========================================
    
    # Check Task A (Task 0) using Final Model
    # 期待: Baseが固定されていて、LoRA[0]が上書きされていなければ、高得点が出るはず
    score_A = evaluate_task(0, path_D)
    
    # Check Task D (Task 3)
    score_D = evaluate_task(3, path_D)
    
    print(f"\nFinal Check:")
    print(f"Task A (Past): {score_A:.1f}")
    print(f"Task D (Curr): {score_D:.1f}")
    
    if score_A > -200: # 閾値は適当
        print("SUCCESS: Task A is preserved!")
    else:
        print("WARNING: Task A might be forgotten or model broken.")


========== Training Task0 _init ==========
[Env] Created SimpleReacherEnv for Task 0 (Target: [1. 1.])
[Logger] Logs will be saved to: result/logs/Reacher_Task0_rank4_init
[Logger] Models will be saved to: result/models/Reacher_Task0_rank4_init
Start training on SimpleReacher (Mac Local / CPU)
Check results in: result
Step: 1000, Loss: 21.072
Eval at step 1000: Reward -173.6
 [CheckPoint] New best model saved at step 1000
Step: 2000, Loss: 0.447
Eval at step 2000: Reward -152.4
 [CheckPoint] New best model saved at step 2000
Step: 3000, Loss: 0.411
Eval at step 3000: Reward -172.4
Step: 4000, Loss: 0.375
Eval at step 4000: Reward -235.4
Step: 5000, Loss: 0.366
Eval at step 5000: Reward -205.4

========== Training Task1 _1st ==========
[Env] Created SimpleReacherEnv for Task 1 (Target: [-1. -1.])
Loading weights from: result/models/Reacher_Task0_rank4_init/best.pth
Freezing Base parameters...
[Logger] Logs will be saved to: result/logs/Reacher_Task1_rank4_1st
[Logger] Models will be sa

KeyboardInterrupt: 